- Widget definitions and parameter passing from workflow

In [0]:
dbutils.widgets.text("topic", "india")
dbutils.widgets.text("page_size","100")
dbutils.widgets.text("language","en")

topic = dbutils.widgets.get("topic")
page_size = int(dbutils.widgets.get("page_size"))
language = dbutils.widgets.get("language")

print (f"Running pipeline for topic: {topic}")
print (f"Page size: {page_size}")
print (f"Language: {language}")

- 1. Call NewsAPI and get articles
- 2. Save the response as a JSON file into volume
- 3. Use Auto Loader to read from the Volume into a delta table

In [0]:
import requests
import json
import os
import time

NEWS_API_KEY = os.environ["NEWS_API_KEY"]
NEWS_API_BASE_URL = os.environ["NEWS_API_BASE_URL"]

if NEWS_API_KEY:
    NEWS_API_KEY = NEWS_API_KEY.strip("'")

if NEWS_API_BASE_URL:
    NEWS_API_BASE_URL = NEWS_API_BASE_URL.strip("'")

URL = f'{NEWS_API_BASE_URL}?q={topic}&language={language}&pageSize={page_size}&apiKey={NEWS_API_KEY}'


max_retries = 3
retry_delay = 5  # seconds

response = None

for attempt in range(max_retries):

    try:
        response = requests.get(URL, timeout=30)

        # Success
        if response.status_code == 200:
            print("API call successful")
            break

        else:
            print(f"Attempt {attempt + 1} failed with status code: {response.status_code}")

    except Exception as e:
        print(f"Attempt {attempt + 1} failed with error: {e}")

    # Exponential backoff — wait longer after each failure
    if attempt < max_retries - 1:
        wait = 2 ** attempt * retry_delay
        print (f"Waiting {wait} seconds before retrying...")
        time.sleep(wait)

# Final validation
if response is None or response.status_code != 200:
    raise Exception("News API request failed after retries")

data = response.json()

In [0]:
print(f"Status: {data['status']}")
print(f"Total results: {data['totalResults']}")
print(f"Articles fetched: {len(data['articles'])}")

- Convert the response dictionary to JSON and write it into the file with current timestamp
- Close the file automatically

In [0]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M")
filename = f"articles_{timestamp}.json"
filepath = f"/Volumes/news_pipeline/bronze/raw_json_files/{filename}"
with open(filepath, 'w') as f:
  for article in data['articles']:
    f.write(json.dumps(article) + '\n')
print(f"Saved {len(data['articles'])} articles to {filepath}")

In [0]:
dbutils.fs.ls("/Volumes/news_pipeline/bronze/raw_json_files")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("author", StringType(), True),
    StructField("content", StringType(), True),
    StructField("description", StringType(), True),
    StructField("publishedAt", StringType(), True),
    StructField("source",StructType([
        StructField("id", StringType(), True),
        StructField("name", StringType(), True)   
    ])),
    StructField("title", StringType(), True),
    StructField("url", StringType(), True),
    StructField("urlToImage", StringType(), True)
])

In [0]:
spark.readStream.format("cloudFiles") \
    .schema(schema) \
    .option("cloudFiles.format","json") \
    .option("cloudFiles.schemaLocation", "/Volumes/news_pipeline/bronze/raw_json_files/_schema") \
    .load("/Volumes/news_pipeline/bronze/raw_json_files") \
    .writeStream.option("checkpointLocation","/Volumes/news_pipeline/bronze/raw_json_files/_checkpoints") \
    .trigger(availableNow=True) \
    .toTable("news_pipeline.bronze.raw_articles")


In [0]:
df = spark.table("news_pipeline.bronze.raw_articles")
df.printSchema()

In [0]:
spark.sql("SELECT * FROM news_pipeline.bronze.raw_articles LIMIT 50").display()